# Part 1 — Does DoRA's weight-decomposition claim hold in a VLM's vision encoder?EN4554 Assignment 1. SmolVLM-256M-Instruct, free Colab T4.**Runtime → Change runtime type → T4 GPU** before running anything.Order of operations:1. Setup + sanity checks (~5 min) — never skip these, they catch silent mis-targeting2. Smoke run (~5 min) — proves the pipeline end to end on 64 examples3. Tier 1 (~2.5 h) — the vision-tower replication of DoRA Figure 2; independently submittable4. Tiers 2–4 — one hypothesis each; run in order, stop when the quota stops you5. Analysis + figures

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader# Colab ships a working torch for this runtime -- do NOT reinstall it.!pip -q install "transformers>=4.46" "peft>=0.13" "datasets>=3.0" accelerate scipy

In [ ]:
import os, sysREPO = "/content/LoRA-finetuning"      # or mount Drive and point hereif not os.path.exists(REPO):    !git clone <YOUR_REPO_URL> {REPO}os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src"))# Durability: results land in Drive so a disconnect costs one run, never the batch.from google.colab import drive; drive.mount('/content/drive')os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/en4554_dora/results"os.environ["FIG_DIR"]     = "/content/drive/MyDrive/en4554_dora/figures"os.makedirs(os.environ["RESULTS_DIR"], exist_ok=True)os.makedirs(os.environ["FIG_DIR"], exist_ok=True)print(os.environ["RESULTS_DIR"])

## 1. Sanity checksThese verify, in order: the dM/dD math; that the magnitude axis is per **output**feature (what PEFT actually trains); that the regex matches all 72 vision Linearsincluding `out_proj`/`fc1`/`fc2`; that magnitude-only has exactly 82,944 trainableparameters; that a DoRA-wrapped model is **bit-identical** to the base model at init;that every image costs exactly 64 tokens; and that the loss covers the answer only.

In [ ]:
!python tests/test_sanity.py

## 2. Smoke run64 train / 32 eval examples. The point is not the numbers — it is that the loss movesand the dM/dD scatter is non-degenerate before any real quota is spent.

In [ ]:
!python run_grid.py --tiers 1 --smoke --methods lora --datasets pathvqa --overwrite

In [ ]:
import json, glob, osrec = json.load(open(sorted(glob.glob(os.path.join(os.environ["RESULTS_DIR"], "lora_V_pathvqa*.json")))[0]))print("loss:", [round(h["loss"], 4) for h in rec["train"]["loss_history"]])print("headline:", rec["accuracy"])print("decomposition rows:", len(rec["decomposition"]))print("sample:", rec["decomposition"][:2])print("predictions:", rec["sample_predictions"][:5])

## 3. Tier 1 — H1 + H3 (6 runs, ~2.5 h)Full FT / LoRA / DoRA on the **vision encoder only**, on PathVQA (domain shift, verifiedabsent from The Cauldron) and TextVQA (in-domain, verified present).Expect accuracy to barely move here: the language model is frozen, so there is no adaptedtask head. That is by design — this tier exists to measure dM/dD under a real gradient.All accuracy claims come from Tier 3.

In [ ]:
!python run_grid.py --tiers 1

## 4. Tier 2 — H2 (vision vs language, same data)

In [ ]:
!python run_grid.py --tiers 2

## 5. Tier 3 — accuracy claims + H5 (includes the seed repeats)

In [ ]:
!python run_grid.py --tiers 3

## 6. Tier 4 — H4, the bonus`magonly` = DoRA with lora_A/lora_B frozen, i.e. a learned per-output-channel gain(82,944 params, 0.03% of the model). `magV_doraL` is DoRA sec. 5.6 / Table 6 translatedto a VLM: magnitude-only in the vision tower, full DoRA in the language model.

In [ ]:
!python run_grid.py --tiers 4

## 7. Analysis and figures

In [ ]:
import importlib, analyzeimportlib.reload(analyze)res = analyze.run_all(os.environ["RESULTS_DIR"])display(res["summary"])display(res["correlations"].query("axis == 'out'")[["run_id","group","r","slope","ft_likeness","headline"]])print(json.dumps(res["h5"], indent=2))

In [ ]:
from IPython.display import Image, display as dfor f in res["figures"]:    print(f); d(Image(f))

## 8. What to write up- **H1** — Fig. 1 top row: do the vision-encoder correlation signs match the paper's  (FT −0.62 / LoRA +0.83 / DoRA −0.31)? Those reference values are printed on each panel.- **H2** — Fig. 1 top vs bottom row. If the language row reproduces the paper's signs but  the vision row does not, that is the finding. If *neither* row reproduces them, suspect  the pipeline before the hypothesis.- **H3** — Fig. 2: is PathVQA shifted right (more direction) relative to TextVQA?- **H4** — Fig. 3: where does the 82,944-parameter point sit on each dataset?- **H5** — `h5.json`. n is 8–12, so report Spearman with its p-value and call it  suggestive. A null result here is the most interesting outcome available at this scale.Also report: the seed, the exact subset sizes and splits, that `do_image_splitting=False`caps image detail, and that the literature check is "searched and did not find", not"does not exist".